In [1]:
from gpt_download import download_and_load_gpt2, load_gpt2_params_from_tf_ckpt
from prev import GPTModel, load_weights_into_gpt
# If the `previous_chapters.py` file is not available locally,
# you can import it from the `llms-from-scratch` PyPI package.
# For details, see: https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# E.g.,
# from llms_from_scratch.ch04 import GPTModel
# from llms_from_scratch.ch05 import download_and_load_gpt2, load_weights_into_gpt
import json, os
import tensorflow as tf


In [2]:
BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-small (124M)"

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(
    model_size=model_size,
    models_dir="gpt2"
)

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval();

File already exists and is up-to-date: gpt2/124M/checkpoint
File already exists and is up-to-date: gpt2/124M/encoder.json
File already exists and is up-to-date: gpt2/124M/hparams.json
File already exists and is up-to-date: gpt2/124M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/124M/model.ckpt.index
File already exists and is up-to-date: gpt2/124M/model.ckpt.meta
File already exists and is up-to-date: gpt2/124M/vocab.bpe


In [3]:
import tiktoken
tokenizer = tiktoken.get_encoding('gpt2')

In [4]:
import os
import requests
import json
from mycode import format_entry

import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader


# url = "https://datasets-server.huggingface.co/rows?dataset=glaiveai%2Fglaive-function-calling-v2&config=default&split=train&offset=0"

# response = requests.get(url, timeout=30)
# response.raise_for_status()
# content = json.loads(response.content)

# data = []
# for row in content["rows"]:
#     data.append(row['row'])

In [5]:
import pandas as pd

# df = pd.read_json("hf://datasets/glaiveai/glaive-function-calling-v2/glaive-function-calling-v2.json")
df = pd.read_json("dataset.json")

In [6]:
data = df.to_dict('records')  

In [7]:
data = data[:10000]

In [8]:
train_portion = int(len(data) * 0.85)
val_portion = int(len(data) * 0.1)
test_portion = len(data) - train_portion - val_portion

train_data = data[:train_portion]
val_data = data[train_portion:train_portion + val_portion]
test_data = data[train_portion + val_portion:]

In [9]:
print(train_portion, val_portion, test_portion)

8500 1000 500


In [11]:
from mycode import format_entry
input_text, response = format_entry(val_data[0])
print(input_text)

###SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "translate_text",
    "description": "Translate text from one language to another",
    "parameters": {
        "type": "object",
        "properties": {
            "text": {
                "type": "string",
                "description": "The text to be translated"
            },
            "source_language": {
                "type": "string",
                "description": "The source language of the text"
            },
            "target_language": {
                "type": "string",
                "description": "The target language for translation"
            }
        },
        "required": [
            "text",
            "source_language",
            "target_language"
        ]
    }
}
###USER: Hi, I have a sentence in French that I need translated to English. The sentence is "Je suis très heureux de vous rencontrer".


In [13]:
from mycode import InstructionDataset, custom_collate_fn
import functools


num_workers = 0
batch_size = 2
allowed_max_length = 512

collate_fn = functools.partial(custom_collate_fn, allowed_max_length=allowed_max_length)

train_dataset   = InstructionDataset(train_data, tokenizer)
val_dataset     = InstructionDataset(val_data, tokenizer)
test_dataset    = InstructionDataset(test_data, tokenizer)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)


In [14]:
from mycode import (
    generate,
    text_to_token_ids,
    token_ids_to_text
)
# Alternatively:
# from llms_from_scratch.ch05 import (
#    generate,
#    text_to_token_ids,
#    token_ids_to_text
# )


token_ids = generate(
    model=model,
    idx=text_to_token_ids(input_text, tokenizer),
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256,
)
generated_text = token_ids_to_text(token_ids, tokenizer)

In [15]:
print(generated_text)

###SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "translate_text",
    "description": "Translate text from one language to another",
    "parameters": {
        "type": "object",
        "properties": {
            "text": {
                "type": "string",
                "description": "The text to be translated"
            },
            "source_language": {
                "type": "string",
                "description": "The source language of the text"
            },
            "target_language": {
                "type": "string",
                "description": "The target language for translation"
            }
        },
        "required": [
            "text",
            "source_language",
            "target_language"
        ]
    }
}
###USER: Hi, I have a sentence in French that I need translated to English. The sentence is "Je suis très heureux de vous rencontrer".

###SYSTEM: You are a helpful assist

In [16]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss

In [17]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total = 0.

    if len(data_loader) == 0:
        return float('nan')
    elif not num_batches:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total += loss
        else:
            break
    
    return total / num_batches


In [18]:
response_text = (
    generated_text[len(input_text):]
    .replace("### Response:", "")
    .strip()
)
print(response_text)

###SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -

{       "name":


In [19]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # Use PyTorch 2.9 or newer for stable mps results
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

print("Device:", device)

Device: mps


In [20]:
model.to(device)

torch.manual_seed(123)

with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: tensor(2.7811, device='mps:0')
Validation loss: tensor(2.6423, device='mps:0')


In [21]:
from mycode import generate_and_print_sample


In [22]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [23]:


def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    # Initialize lists to track losses and tokens seen
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    # Main training loop
    for epoch in range(num_epochs):
        model.train()  # Set model to training mode

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()  # Reset loss gradients from previous batch iteration
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()  # Calculate loss gradients
            optimizer.step()  # Update model weights using loss gradients
            tokens_seen += input_batch.numel()
            global_step += 1

            # Optional evaluation step
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # Print a sample text after each epoch
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen

In [24]:
import time

start_time = time.time()

torch.manual_seed(123)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.1)

num_epochs = 2

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=format_entry(val_data[0])[0], tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

Ep 1 (Step 000000): Train loss 2.097, Val loss 2.016
Ep 1 (Step 000005): Train loss 0.943, Val loss 1.078
Ep 1 (Step 000010): Train loss 0.617, Val loss 0.780
Ep 1 (Step 000015): Train loss 0.535, Val loss 0.735
Ep 1 (Step 000020): Train loss 0.473, Val loss 0.702
Ep 1 (Step 000025): Train loss 0.467, Val loss 0.663
Ep 1 (Step 000030): Train loss 0.481, Val loss 0.638
Ep 1 (Step 000035): Train loss 0.469, Val loss 0.650
Ep 1 (Step 000040): Train loss 0.415, Val loss 0.642
Ep 1 (Step 000045): Train loss 0.465, Val loss 0.649
Ep 1 (Step 000050): Train loss 0.406, Val loss 0.599
Ep 1 (Step 000055): Train loss 0.374, Val loss 0.572
Ep 1 (Step 000060): Train loss 0.353, Val loss 0.522
Ep 1 (Step 000065): Train loss 0.350, Val loss 0.501
Ep 1 (Step 000070): Train loss 0.332, Val loss 0.490
Ep 1 (Step 000075): Train loss 0.317, Val loss 0.493
Ep 1 (Step 000080): Train loss 0.305, Val loss 0.521
Ep 1 (Step 000085): Train loss 0.300, Val loss 0.548
Ep 1 (Step 000090): Train loss 0.281, Val loss

RuntimeError: MPS backend out of memory (MPS allocated: 9.19 GiB, other allocations: 10.78 GiB, max allowed: 20.13 GiB). Tried to allocate 195.93 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [ ]:
torch.manual_seed(123)


for i, entry in enumerate(test_data[:10]):
    print(f'=========== {i} ===========\n')

    input_text, output_text = format_entry(entry)

    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    print(input_text)
    print(f"\nCorrect response:\n>> {output_text}")
    print(f"\nModel response:\n>> {response_text.strip()}")
    print("-------------------------------------")

=========== 0 ===========

###SYSTEM: You are a helpful assistant with access to the following functions. Use them if required -
{
    "name": "get_random_quote",
    "description": "Get a random quote",
    "parameters": {}
}

{
    "name": "get_stock_price",
    "description": "Get the current stock price",
    "parameters": {
        "type": "object",
        "properties": {
            "symbol": {
                "type": "string",
                "description": "The stock symbol"
            }
        },
        "required": [
            "symbol"
        ]
    }
}

###USER: I need some inspiration. Can you give me a quote?

Correct response:
>> ###ASSISTANT: <functioncall> {"name": "get_random_quote", "arguments": {}} 

Model response:
>> ###ASSISTANT: <functioncall> {"name": "get_stock_price", "arguments": '{"symbol": "AAPL"}'}
-------------------------------------
=========== 1 ===========

###SYSTEM: You are a helpful assistant with access to the following functions. Use them if